# CyberSentinel — NSL-KDD Threat Detection System
### Big Data Analytics Project

**How to use this notebook:**
1. Run cells **in order** from top to bottom
2. In **Cell 2**, upload your train and test files
3. Wait for training to finish (Cell 3–6)
4. Run Cell 7 to launch the live dashboard — a public URL will appear
5. Open that URL in any browser tab to see your dashboard

---
**Files needed from NSL-KDD dataset:**
- `KDDTrain+.txt` — for training the model
- `KDDTest+.txt`  — for testing and live simulation

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Install required libraries
# This installs everything we need. Run once, takes ~1 minute.
# ════════════════════════════════════════════════════════════════

!pip install streamlit plotly pyngrok -q

# Check versions so we know everything installed correctly
import streamlit, plotly, pyngrok
print(' streamlit:', streamlit.__version__)
print(' plotly:',     plotly.__version__)
print(' pyngrok:',    pyngrok.__version__)
print('\nAll libraries ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.6 MB/s eta 0:00:00
 streamlit: 1.57.0
 plotly: 5.24.1
 pyngrok: 8.1.2

All libraries ready!


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Upload your NSL-KDD dataset files
# Upload BOTH:
#   • KDDTrain+.txt
#   • KDDTest+.txt
# ════════════════════════════════════════════════════════════════

from google.colab import files

print(' Please upload your KDDTrain+.txt and KDDTest+.txt files...')
uploaded = files.upload()

# Confirm what was uploaded
print('\n Files uploaded:')
for name in uploaded:
    size_kb = len(uploaded[name]) / 1024
    print(f'   {name}  ({size_kb:.0f} KB)')

# Make sure we got the right files
import os
for needed in ['KDDTrain+.txt', 'KDDTest+.txt']:
    if needed in uploaded:
        print(f' {needed} — found')
    else:
        print(f' {needed} — MISSING! Please re-upload.')

 Please upload your KDDTrain+.txt and KDDTest+.txt files...


Saving KDDTest+.txt to KDDTest+.txt
Saving KDDTrain+.txt to KDDTrain+.txt

 Files uploaded:
   KDDTest+.txt  (3361 KB)
   KDDTrain+.txt  (18662 KB)
 KDDTrain+.txt — found
 KDDTest+.txt — found


In [ ]:
import pandas as pd
import numpy as np
import os

# These are the official NSL-KDD column names in order
COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
    'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login',
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty_level'
]

# Dynamically find the correct filenames in the current directory
train_filename = None
test_filename = None

for f in os.listdir('.'):
    if 'KDDTrain+' in f and '.txt' in f:
        train_filename = f
    if 'KDDTest+' in f and '.txt' in f:
        test_filename = f

if not train_filename:
    raise FileNotFoundError("Could not find KDDTrain+.txt or its renamed version. Please ensure it was uploaded in Cell 2.")
if not test_filename:
    raise FileNotFoundError("Could not find KDDTest+.txt or its renamed version. Please ensure it was uploaded in Cell 2.")

# header=None tells pandas this file has no header row
# names=COLUMNS assigns our column names
train_df = pd.read_csv(train_filename, header=None, names=COLUMNS)
test_df  = pd.read_csv(test_filename,  header=None, names=COLUMNS)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Training set : {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'Test set     : {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# Show how many of each attack type exist in training data
print('\nLabel distribution in training set:')
label_counts = train_df['label'].value_counts()
for label, count in label_counts.items():
    bar = '█' * min(40, int(count / label_counts.max() * 40))
    print(f'  {label:20s} {count:6,}  {bar}')

print('\nFirst 3 rows of training data:')
train_df.head(3)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Training set : 125,973 rows × 43 columns
Test set     : 22,544 rows × 43 columns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Label distribution in training set:
  normal               67,343  ████████████████████████████████████████
  neptune              41,214  ████████████████████████
  satan                 3,633  ██
  ipsweep               3,599  ██
  portsweep             2,931  █
  smurf                 2,646  █
  nmap                  1,493  
  back                    956  
  teardrop                892  
  warezclient             890  
  pod                     201  
  guess_passwd             53  
  buffer_overflow          30  
  warezmaster              20  
  land                     18  
  imap                     11  
  rootkit                  10  
  loadmodule                9  
  ftp_write                 8  
  multihop                  7  
  phf                       4  
  perl                      3  
  spy                     

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,neptune,19


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Preparing data for machine learning
#
#   Creating binary labels (normal=0, attack=1)
#   Encoding text columns as numbers
#   Normalizing all numbers to the same scale
# ════════════════════════════════════════════════════════════════

from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Step 1: Binary labels ────────────────────────────────────────
# NSL-KDD has many attack names (neptune, smurf, portsweep...)
# We simplify: normal=0, any attack=1
train_df['is_attack'] = (train_df['label'] != 'normal').astype(int)
test_df['is_attack']  = (test_df['label']  != 'normal').astype(int)

normal_count = (train_df['is_attack'] == 0).sum()
attack_count = (train_df['is_attack'] == 1).sum()
print(f'Training set — Normal: {normal_count:,}  |  Attacks: {attack_count:,}')

# ── Step 2: Encode text columns ──────────────────────────────────
# ML models only understand numbers.

le = LabelEncoder()
text_cols = ['protocol_type', 'service', 'flag']

for col in text_cols:
    le.fit(train_df[col])                # Learn categories from train
    train_df[col] = le.transform(train_df[col])
    # Test may contain new categories not in train — map those to -1
    test_df[col] = test_df[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    print(f'  Encoded column: {col}')

# ── Step 3: Separate features from labels ────────────────────────
drop_cols = ['label', 'difficulty_level', 'is_attack']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_attack']
X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df['is_attack']

# ── Step 4: Normalize ────────────────────────────────────────────
# src_bytes can be in millions; duration can be 0.001
# StandardScaler transforms all columns to mean=0, std=1
# This stops large-scale columns from dominating the model

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Learn stats from train
X_test_scaled  = scaler.transform(X_test)        # Apply same stats to test

print(f'\n Features ready: {X_train.shape[1]} columns')
print(f' X_train scaled shape: {X_train_scaled.shape}')
print(f' X_test  scaled shape: {X_test_scaled.shape}')

Training set — Normal: 67,343  |  Attacks: 58,630
  Encoded column: protocol_type
  Encoded column: service
  Encoded column: flag

 Features ready: 41 columns
 X_train scaled shape: (125973, 41)
 X_test  scaled shape: (22544, 41)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Train the Isolation Forest model
#
# Isolation Forest logic:
#   - Train ONLY on normal traffic
#   - The model learns what "normal" looks like
#   - Anything that doesn't fit = anomaly = possible attack
#   - It works by randomly isolating data points in trees.
#     Normal points need many splits to isolate.
#     Anomalies get isolated very quickly = flagged.
# ════════════════════════════════════════════════════════════════

from sklearn.ensemble import IsolationForest

# Extract ONLY the normal rows for training
# The model should learn only normal behavior
normal_train = X_train_scaled[y_train == 0]
print(f'Training on {normal_train.shape[0]:,} normal-only rows...')

iso_model = IsolationForest(
    n_estimators=100,    # 100 decision trees — more = more accurate but slower
    contamination=0.05,  # We expect ~5% of data to be anomalous
    random_state=42      # Fixed seed = same result every run
)
iso_model.fit(normal_train)
print(' Isolation Forest trained!')

# ── Rule-based detection function ────────────────────────────────
# These rules are based on known NSL-KDD attack signatures.
# Each rule targets a specific attack pattern.

def apply_rules(df):
    """
    Takes a DataFrame, returns array of 0s and 1s.
    1 = this row triggered at least one rule.
    """
    flags = np.zeros(len(df), dtype=int)

    # Rule 1: Very high source bytes → DDoS or data exfiltration
    flags[df['src_bytes'] > df['src_bytes'].quantile(0.99)] = 1

    # Rule 2: Many connections to same destination → port sweep
    flags[df['dst_host_count'] > 250] = 1

    # Rule 3: High SYN error rate → SYN flood attack
    # serror_rate = fraction of connections with SYN errors
    flags[df['serror_rate'] > 0.8] = 1

    # Rule 4: root_shell = 1 means attacker got root/admin access
    flags[df['root_shell'] == 1] = 1

    # Rule 5: Failed logins → brute force password attack
    flags[df['num_failed_logins'] > 0] = 1

    return flags

# Run both detection layers on test set
rule_flags = apply_rules(X_test)
ml_preds   = iso_model.predict(X_test_scaled)
ml_flags   = (ml_preds == -1).astype(int)  # -1 = anomaly in IsolationForest

# Combine: flagged by EITHER layer = threat
final_flags = ((rule_flags == 1) | (ml_flags == 1)).astype(int)

print(f'\nRule layer  flagged: {rule_flags.sum():,} entries')
print(f'ML layer    flagged: {ml_flags.sum():,} entries')
print(f'Combined    flagged: {final_flags.sum():,} entries')
print(f'Out of total       : {len(final_flags):,} entries')

Training on 67,343 normal-only rows...
 Isolation Forest trained!

Rule layer  flagged: 15,123 entries
ML layer    flagged: 8,507 entries
Combined    flagged: 15,881 entries
Out of total       : 22,544 entries


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — Evaluate model performance
#
# We compare our predictions against the real labels.
# Key metrics:
#   Precision = of everything we flagged, how many were real attacks?
#   Recall    = of all real attacks, how many did we catch?
#   F1        = balanced average of precision and recall
#   Accuracy  = overall correct predictions / total
# ════════════════════════════════════════════════════════════════

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, final_flags).ravel()

# Calculate metrics manually so we understand the formula
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy  = (tp + tn) / (tp + tn + fp + fn)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('         CONFUSION MATRIX')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  True  Positives (attacks caught)   : {tp:,}')
print(f'  True  Negatives (normal, correct)  : {tn:,}')
print(f'  False Positives (false alarms)     : {fp:,}')
print(f'  False Negatives (missed attacks)   : {fn:,}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Precision : {precision:.4f}  ({precision*100:.1f}%)')
print(f'  Recall    : {recall:.4f}  ({recall*100:.1f}%)')
print(f'  F1 Score  : {f1:.4f}')
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# Attack category breakdown
# These patterns match known NSL-KDD attack names to categories
ATTACK_MAP = {
    'DoS'  : ['neptune','smurf','pod','teardrop','land','back','apache2','udpstorm','processtable','mailbomb'],
    'Probe': ['portsweep','ipsweep','nmap','satan','saint','mscan'],
    'R2L'  : ['guess_passwd','ftp_write','imap','phf','multihop','warezmaster','warezclient','spy','xlock','xsnoop','snmpgetattack','named','sendmail','httptunnel'],
    'U2R'  : ['buffer_overflow','perl','rootkit','loadmodule','sqlattack','xterm','ps']
}

print('\nAttack category breakdown in test set:')
for cat, names in ATTACK_MAP.items():
    count = test_df[test_df['label'].isin(names)].shape[0]
    detected = test_df[
        test_df['label'].isin(names)
    ].index
    det_count = final_flags[detected].sum() if len(detected) > 0 else 0
    print(f'  {cat:6s} — Total: {count:5,}  Detected: {det_count:5,}')

# Save metrics for the dashboard to read
import json
metrics = {
    'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    'precision': round(precision, 4), 'recall': round(recall, 4),
    'f1': round(f1, 4), 'accuracy': round(accuracy, 4),
    'total': int(len(y_test)),
    'threats': int(final_flags.sum()),
    'normal_count': int((final_flags == 0).sum())
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f)
print('\n Metrics saved to metrics.json')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         CONFUSION MATRIX
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  True  Positives (attacks caught)   : 11,521
  True  Negatives (normal, correct)  : 5,351
  False Positives (false alarms)     : 4,360
  False Negatives (missed attacks)   : 1,312
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Precision : 0.7255  (72.5%)
  Recall    : 0.8978  (89.8%)
  F1 Score  : 0.8025
  Accuracy  : 0.7484  (74.8%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Attack category breakdown in test set:
  DoS    — Total: 7,458  Detected: 7,357
  Probe  — Total: 2,421  Detected: 2,175
  R2L    — Total: 2,554  Detected: 1,622
  U2R    — Total:    67  Detected:    39

 Metrics saved to metrics.json


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Write the Streamlit dashboard app to a file
# This cell writes dashboard.py onto disk inside Colab.
# The %%writefile magic command does the writing.
# ════════════════════════════════════════════════════════════════

dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import json, os, time
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix

# ── Page configuration ───────────────────────────────────────────
st.set_page_config(
    page_title="CyberSentinel IDS",
    page_icon="shield",
    layout="wide"
)

# ── Dark terminal styling ─────────────────────────────────────────
st.markdown("""
<style>
    .stApp { background-color: #060a10; }
    .block-container { padding-top: 1.5rem; }
    h1,h2,h3,p,div { color: #c8daf0 !important; }
    .metric-card {
        background: #0b1220;
        border: 1px solid #1a2a40;
        border-top: 2px solid #00d4ff;
        border-radius: 6px;
        padding: 16px 20px;
        text-align: center;
    }
    .metric-val  { font-family: monospace; font-size: 32px; font-weight: 700; line-height:1; }
    .metric-lbl  { font-size: 11px; letter-spacing: 2px; color: #4a6282;
                   text-transform: uppercase; margin-top: 4px; }
    .threat-row  { background: rgba(255,59,92,0.08);
                   border-left: 3px solid #ff3b5c;
                   padding: 4px 10px; margin: 2px 0;
                   font-family: monospace; font-size: 12px;
                   color: #ff3b5c; border-radius: 0 4px 4px 0; }
    .normal-row  { background: rgba(0,230,118,0.04);
                   border-left: 3px solid #00e676;
                   padding: 4px 10px; margin: 2px 0;
                   font-family: monospace; font-size: 12px;
                   color: #00e676; border-radius: 0 4px 4px 0; }
    .alert-banner { background: rgba(255,59,92,0.15);
                    border: 1px solid #ff3b5c; border-radius: 6px;
                    padding: 12px 16px; margin-bottom: 10px;
                    font-family: monospace; font-size: 13px;
                    color: #ff3b5c; }
    .section-title { font-size: 11px; letter-spacing: 3px;
                     text-transform: uppercase; color: #4a6282 !important;
                     margin-bottom: 8px; }
    [data-testid="stMetricValue"] { color: #00d4ff !important; }
    [data-testid="stMetricLabel"] { color: #4a6282 !important; }
</style>
""", unsafe_allow_html=True)

# ── Constants ─────────────────────────────────────────────────────
COLUMNS = [
    'duration','protocol_type','service','flag',
    'src_bytes','dst_bytes','land','wrong_fragment',
    'urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files',
    'num_outbound_cmds','is_host_login','is_guest_login',
    'count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate',
    'dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty_level'
]

ATTACK_MAP = {
    'neptune':'DoS','smurf':'DoS','pod':'DoS','teardrop':'DoS',
    'land':'DoS','back':'DoS','apache2':'DoS',
    'portsweep':'Probe','ipsweep':'Probe','nmap':'Probe','satan':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','multihop':'R2L',
    'buffer_overflow':'U2R','perl':'U2R','rootkit':'U2R','loadmodule':'U2R',
    'normal':'Normal'
}

def safe_div(a, b):
    return round(a / b, 4) if b > 0 else 0

# ── Load + train (cached so it only runs once per session) ────────
@st.cache_resource
def load_and_train():
    """Load data, train model. @st.cache_resource means this
       only runs ONCE even if the page refreshes."""
    train_df = pd.read_csv("KDDTrain+.txt", header=None, names=COLUMNS)
    test_df  = pd.read_csv("KDDTest+.txt",  header=None, names=COLUMNS)

    train_df["is_attack"] = (train_df["label"] != "normal").astype(int)
    test_df["is_attack"]  = (test_df["label"]  != "normal").astype(int)

    le = LabelEncoder()
    for col in ["protocol_type", "service", "flag"]:
        le.fit(train_df[col])
        train_df[col] = le.transform(train_df[col])
        test_df[col]  = test_df[col].map(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )

    drop = ["label", "difficulty_level", "is_attack"]
    X_train = train_df.drop(columns=drop)
    y_train = train_df["is_attack"]
    X_test  = test_df.drop(columns=drop)
    y_test  = test_df["is_attack"]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    model.fit(X_train_s[y_train == 0])

    return test_df, X_test, X_test_s, y_test, scaler, model

def apply_rules(df):
    flags = np.zeros(len(df), dtype=int)
    flags[df["src_bytes"] > df["src_bytes"].quantile(0.99)] = 1
    flags[df["dst_host_count"] > 250]  = 1
    flags[df["serror_rate"] > 0.8]     = 1
    flags[df["root_shell"] == 1]        = 1
    flags[df["num_failed_logins"] > 0]  = 1
    return flags

# ── Session state — persists across reruns ────────────────────────
# st.session_state is like a dictionary that survives page refreshes.
# We use it to track which packet we are on during simulation.
if "idx" not in st.session_state:
    st.session_state.idx      = 0
    st.session_state.running  = False
    st.session_state.log      = []
    st.session_state.tp       = 0
    st.session_state.tn       = 0
    st.session_state.fp       = 0
    st.session_state.fn       = 0
    st.session_state.normal   = 0
    st.session_state.threats  = 0
    st.session_state.timeline = []
    st.session_state.attacks  = {"DoS":0, "Probe":0, "R2L":0, "U2R":0}

# ── Header ────────────────────────────────────────────────────────
col_title, col_status = st.columns([3, 1])
with col_title:
    st.markdown("## CyberSentinel IDS")
with col_status:
    st.markdown("<br>", unsafe_allow_html=True)
    status = "LIVE" if st.session_state.running else "PAUSED"
    color  = "#00e676" if st.session_state.running else "#ffb300"
    st.markdown(f'<div style="text-align:right;font-family:monospace;color:{color};font-size:13px;">● {status}</div>', unsafe_allow_html=True)

st.markdown("---")

# ── Load data and train ───────────────────────────────────────────
with st.spinner("Loading NSL-KDD and training model (first run only)..."):
    test_df, X_test, X_test_s, y_test, scaler, model = load_and_train()

TOTAL_ROWS = len(X_test)

# ── Controls ──────────────────────────────────────────────────────
c1, c2, c3, c4 = st.columns([1, 1, 1, 3])
with c1:
    if st.button("▶ Start" if not st.session_state.running else "⏸ Pause"):
        st.session_state.running = not st.session_state.running
with c2:
    if st.button("↺ Reset"):
        for key in ["idx","log","tp","tn","fp","fn","normal","threats","timeline"]:
            st.session_state[key] = 0 if key == "idx" else [] if key in ["log","timeline"] else 0
        st.session_state.attacks  = {"DoS":0, "Probe":0, "R2L":0, "U2R":0}
        st.session_state.running  = False
with c3:
    speed = st.selectbox("Speed", ["Slow (50/batch)","Normal (100/batch)","Fast (200/batch)"], index=1)
    batch = {"Slow (50/batch)":50, "Normal (100/batch)":100, "Fast (200/batch)":200}[speed]
with c4:
    st.markdown(f"<small style='color:#4a6282'>Analyzing packet {st.session_state.idx:,} of {TOTAL_ROWS:,}</small>", unsafe_allow_html=True)
    st.progress(min(st.session_state.idx / TOTAL_ROWS, 1.0))

# ── Process next batch of packets ─────────────────────────────────
# Each time Streamlit reruns (every second), we process `batch` more rows.
# This simulates a real-time stream at a controlled speed.
if st.session_state.running and st.session_state.idx < TOTAL_ROWS:
    end_idx = min(st.session_state.idx + batch, TOTAL_ROWS)
    rule_flags = apply_rules(X_test.iloc[st.session_state.idx:end_idx])
    ml_preds   = model.predict(X_test_s[st.session_state.idx:end_idx])
    ml_flags   = (ml_preds == -1).astype(int)
    combined   = ((rule_flags == 1) | (ml_flags == 1)).astype(int)
    actuals    = y_test.iloc[st.session_state.idx:end_idx].values

    for i, (pred, actual) in enumerate(zip(combined, actuals)):
        row_idx   = st.session_state.idx + i
        label     = test_df["label"].iloc[row_idx]
        category  = ATTACK_MAP.get(label, "Other")
        layer     = "RULE" if rule_flags[i] else ("ML" if ml_flags[i] else "PASS")
        src_bytes = int(X_test["src_bytes"].iloc[row_idx])
        duration  = float(X_test["duration"].iloc[row_idx])

        if pred and actual:       st.session_state.tp += 1
        elif not pred and not actual: st.session_state.tn += 1
        elif pred and not actual:     st.session_state.fp += 1
        else:                         st.session_state.fn += 1

        if pred: st.session_state.threats += 1
        else:    st.session_state.normal  += 1

        if pred and category in st.session_state.attacks:
            st.session_state.attacks[category] += 1

        # Keep last 15 log entries for display
        entry = f"[{time.strftime('%H:%M:%S')}] [{layer}] {label.upper():20s} src={src_bytes:>8,}B  dur={duration:.3f}s"
        st.session_state.log.insert(0, (entry, bool(pred)))
        st.session_state.log = st.session_state.log[:15]

    # Add timeline point
    st.session_state.timeline.append({
        "t": time.strftime("%H:%M:%S"),
        "n": st.session_state.normal,
        "th": st.session_state.threats
    })
    st.session_state.timeline = st.session_state.timeline[-40:]
    st.session_state.idx = end_idx

# ── Derived metrics ───────────────────────────────────────────────
tp, tn, fp, fn = st.session_state.tp, st.session_state.tn, st.session_state.fp, st.session_state.fn
total    = st.session_state.idx
threats  = st.session_state.threats
precision= safe_div(tp, tp + fp)
recall   = safe_div(tp, tp + fn)
f1       = safe_div(2 * precision * recall, precision + recall)
accuracy = safe_div(tp + tn, total)

# ── Metric cards ──────────────────────────────────────────────────
st.markdown("<div class='section-title'>System Metrics</div>", unsafe_allow_html=True)
mc1, mc2, mc3, mc4, mc5 = st.columns(5)
def metric_card(col, val, lbl, color="#00d4ff"):
    col.markdown(f"""
        <div class="metric-card">
            <div class="metric-val" style="color:{color}">{val}</div>
            <div class="metric-lbl">{lbl}</div>
        </div>""", unsafe_allow_html=True)

metric_card(mc1, f"{total:,}",            "Packets Analyzed",  "#00d4ff")
metric_card(mc2, f"{st.session_state.normal:,}", "Normal Traffic",  "#00e676")
metric_card(mc3, f"{threats:,}",           "Threats Detected",  "#ff3b5c")
metric_card(mc4, f"{precision*100:.1f}%",  "Precision",         "#ffb300")
metric_card(mc5, f"{recall*100:.1f}%",     "Recall",            "#b48eff")

st.markdown("<br>", unsafe_allow_html=True)

# ── Alert banner for active threats ───────────────────────────────
if threats > 0 and st.session_state.running:
    last_threat = next((e for e, is_t in st.session_state.log if is_t), None)
    if last_threat:
        st.markdown(f'<div class="alert-banner">⚠ THREAT DETECTED — {last_threat}</div>', unsafe_allow_html=True)

# ── Charts row ────────────────────────────────────────────────────
chart_l, chart_r = st.columns([2, 1])

with chart_l:
    st.markdown("<div class='section-title'>Live Traffic Timeline</div>", unsafe_allow_html=True)
    if st.session_state.timeline:
        tl = st.session_state.timeline
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=[p["t"] for p in tl], y=[p["n"] for p in tl],
            name="Normal", line=dict(color="#00e676", width=2),
            fill="tozeroy", fillcolor="rgba(0,230,118,0.06)"
        ))
        fig.add_trace(go.Scatter(
            x=[p["t"] for p in tl], y=[p["th"] for p in tl],
            name="Threats", line=dict(color="#ff3b5c", width=2),
            fill="tozeroy", fillcolor="rgba(255,59,92,0.08)"
        ))
        fig.update_layout(
            paper_bgcolor="#0b1220", plot_bgcolor="#060a10",
            font=dict(color="#c8daf0", size=11), height=220,
            margin=dict(l=0, r=0, t=10, b=0),
            legend=dict(orientation="h", y=1.15, bgcolor="rgba(0,0,0,0)"),
            xaxis=dict(showgrid=True, gridcolor="#1a2a40"),
            yaxis=dict(showgrid=True, gridcolor="#1a2a40")
        )
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.markdown("<small style='color:#4a6282'>Start simulation to see timeline...</small>", unsafe_allow_html=True)

with chart_r:
    st.markdown("<div class='section-title'>Attack Categories</div>", unsafe_allow_html=True)
    ac = st.session_state.attacks
    fig2 = go.Figure(go.Bar(
        x=list(ac.values()), y=list(ac.keys()),
        orientation="h",
        marker_color=["#ff3b5c", "#ffb300", "#b48eff", "#00d4ff"],
        text=list(ac.values()), textposition="outside",
        textfont=dict(color="#c8daf0", size=11)
    ))
    fig2.update_layout(
        paper_bgcolor="#0b1220", plot_bgcolor="#060a10",
        font=dict(color="#c8daf0", size=11), height=220,
        margin=dict(l=0, r=30, t=10, b=0),
        xaxis=dict(showgrid=True, gridcolor="#1a2a40")
    )
    st.plotly_chart(fig2, use_container_width=True)

# ── Confusion matrix + performance ────────────────────────────────
st.markdown("<div class='section-title'>Model Evaluation</div>", unsafe_allow_html=True)
mat_col, perf_col = st.columns([1, 2])

with mat_col:
    mat_df = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual Normal", "Actual Attack"],
        columns=["Predicted Normal", "Predicted Attack"]
    )
    st.dataframe(mat_df, use_container_width=True)

with perf_col:
    p1, p2, p3, p4 = st.columns(4)
    p1.metric("Accuracy",  f"{accuracy*100:.1f}%")
    p2.metric("F1 Score",  f"{f1:.3f}")
    p3.metric("False Pos.", f"{fp:,}")
    p4.metric("Missed Att.", f"{fn:,}")
    st.markdown("""
        <small style='color:#4a6282'>
        Precision = of everything flagged, how many were real attacks<br>
        Recall = of all real attacks, how many did we catch<br>
        F1 = balanced score combining both — higher is better<br>
        False Negatives (missed) matter most in security systems
        </small>""", unsafe_allow_html=True)

# ── Live event log ─────────────────────────────────────────────────
st.markdown("<div class='section-title'>Live Event Feed</div>", unsafe_allow_html=True)
for entry, is_threat in st.session_state.log:
    css = "threat-row" if is_threat else "normal-row"
    icon = "THREAT" if is_threat else "OK"
    st.markdown(f'<div class="{css}">[{icon}] {entry}</div>', unsafe_allow_html=True)

# ── Auto-rerun every 1 second while simulation is running ──────────
# st.rerun() makes Streamlit re-execute this whole script,
# which processes the next batch and refreshes the display.
if st.session_state.running and st.session_state.idx < TOTAL_ROWS:
    time.sleep(1)
    st.rerun()
elif st.session_state.idx >= TOTAL_ROWS:
    st.success(f"Simulation complete! Processed all {TOTAL_ROWS:,} packets.")
    st.session_state.running = False
'''

# Write the dashboard code to a file
with open('dashboard.py', 'w') as f:
    f.write(dashboard_code)

print('✅ dashboard.py written successfully!')
print(f'   Size: {len(dashboard_code):,} characters')

✅ dashboard.py written successfully!
   Size: 15,340 characters


In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

# Kill any existing Streamlit processes to avoid port conflicts
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(1)

# Start Streamlit in a background thread
# --server.headless=true  → no browser pop-up inside Colab
# --server.port 8501      → use port 8501
# --server.enableCORS false → allow connections from ngrok
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'dashboard.py',
        '--server.headless=true',
        '--server.port', '8501',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false'
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Wait a few seconds for Streamlit to fully start up
print('Starting Streamlit...')
time.sleep(5)

# Create the public tunnel
# If you have an ngrok account, paste your auth token below for
# a more stable connection. Free tier works fine for presentations.
# IMPORTANT: Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
# and replace "YOUR_TOKEN_HERE" with it.
#ngrok.set_auth_token("Enter Token")
public_url = ngrok.connect(8501)

print('\n' + '='*55)
print('   DASHBOARD IS LIVE!')
print('='*55)
print(f'  Open this URL in your browser:')
print(f'  {public_url}')
print('='*55)
print('\n  HOW TO USE:')
print('  1. Open the URL above in a new browser tab')
print('  2. Click "▶ Start" to begin the live simulation')
print('  3. Watch threats get detected in real time!')
print('\n  Keep this cell running — closing it stops the dashboard.')
print('  The URL stays active for ~2 hours on free ngrok.')

# Keep the cell alive so the tunnel stays open
# Press the stop button on this cell to shut down the dashboard
try:
    while True:
        time.sleep(30)
        print(f'[{time.strftime("%H:%M:%S")}] Dashboard running... URL: {public_url}')
except KeyboardInterrupt:
    ngrok.kill()
    print('Dashboard stopped.')

ModuleNotFoundError: No module named 'pyngrok'


# ReadMe
## Understanding the Detection System

### Layer 1 — Rule-Based Detection
Manually defined thresholds based on known NSL-KDD attack signatures:
- `src_bytes > 99th percentile` → possible DDoS or data exfiltration
- `dst_host_count > 250` → possible port sweep (Probe attack)
- `serror_rate > 0.8` → SYN flood (DoS attack)
- `root_shell = 1` → attacker gained root access (U2R attack)
- `num_failed_logins > 0` → brute force attempt (R2L attack)

### Layer 2 — Isolation Forest (ML)
- Trains only on **normal traffic** — learns what normal looks like
- Any packet that doesn't fit the normal pattern = anomaly = flagged
- Catches attack patterns rules might miss

### Combined Result
- Flagged by **either** layer = threat
- Two layers working together = higher recall (fewer missed attacks)

---
## NSL-KDD Attack Categories

| Category | What it is | Example attacks |
|----------|-----------|----------------|
| DoS | Denial of Service — flood server to crash it | neptune, smurf, pod |
| Probe | Scanning network to find vulnerabilities | portsweep, ipsweep, nmap |
| R2L | Remote attacker gains local access | guess_passwd, ftp_write |
| U2R | Local user gains root/admin access | buffer_overflow, rootkit |